# Arize Singapore Workshop: Trace & Evaluate a LangGraph Agent

In this notebook you will:
1. Build a **customer-support agent** for a fictional retailer (Sunrise Outfitters) with **LangGraph** + **OpenAI**.
2. **Trace** it into **Arize** with one-line auto-instrumentation.
3. Chat with it through a **Gradio** UI (inline, right here in Colab).
4. **Evaluate** it two ways: online LLM-as-a-judge in the Arize UI, and an offline experiment via the `ax` CLI.

**The only input you need to start is an OpenAI API key.** Tracing later also needs an Arize Space ID + API key (free at [app.arize.com](https://app.arize.com)).

## Step 1 - Install dependencies

In [ ]:
%pip install -q -U \
    langgraph \
    langchain \
    langchain-openai \
    arize-otel \
    openinference-instrumentation-langchain \
    gradio \
    openai

## Step 2 - Add your OpenAI API key

This is the only credential needed to get the agent running.

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")

## Step 3 - Define the support tools

Mock tools backed by in-memory data so we need no real backend. The agent will call these to look up orders, check refund eligibility, search the FAQ, and escalate to a human.

In [ ]:
from langchain_core.tools import tool

_ORDERS = {
    "A1001": {"status": "shipped", "item": "Trailblazer Rain Jacket (M, Forest Green)",
               "carrier": "DHL", "tracking": "DHL-SG-77123", "ordered_days_ago": 3,
               "delivered": False, "price_usd": 129.00},
    "A1002": {"status": "delivered", "item": "Summit Hiking Boots (US 9)",
               "carrier": "SingPost", "tracking": "SP-99812", "ordered_days_ago": 20,
               "delivered": True, "price_usd": 159.00},
    "A1003": {"status": "processing", "item": "Coastline Windbreaker (L, Navy)",
               "carrier": None, "tracking": None, "ordered_days_ago": 1,
               "delivered": False, "price_usd": 89.00},
}

_FAQ = [
    (["shipping", "ship", "delivery", "deliver", "how long", "arrive"],
     "Standard shipping within Singapore takes 2-4 business days. International orders take 7-14 business days. You get a tracking number by email once it ships."),
    (["return", "returns", "refund", "exchange", "money back"],
     "You can return unworn items within 30 days of delivery for a full refund. Items must have original tags."),
    (["size", "sizing", "fit", "measurements", "chart"],
     "Jackets run true to size; boots run about half a size large. A size chart is on every product page under 'Size & Fit'."),
    (["payment", "pay", "card", "paynow", "installment"],
     "We accept all major credit cards, PayNow, and Atome installments. Payment is charged when your order ships."),
]


@tool
def lookup_order(order_id: str) -> str:
    """Look up the status and details of a customer order by its ID (e.g. 'A1001')."""
    order = _ORDERS.get(order_id.strip().upper())
    if not order:
        return f"No order found with ID '{order_id}'. Ask the customer to double-check it."
    parts = [f"Order {order_id.upper()}: {order['item']}", f"Status: {order['status']}",
             f"Placed: {order['ordered_days_ago']} day(s) ago", f"Price: ${order['price_usd']:.2f}"]
    if order["tracking"]:
        parts.append(f"Carrier: {order['carrier']} (tracking {order['tracking']})")
    return ". ".join(parts) + "."


@tool
def check_refund_eligibility(order_id: str) -> str:
    """Check whether an order is eligible for a refund (delivered items: within 30 days)."""
    order = _ORDERS.get(order_id.strip().upper())
    if not order:
        return f"No order found with ID '{order_id}', so eligibility cannot be checked."
    if order["status"] == "processing":
        return f"Order {order_id.upper()} is still processing and can be cancelled now for a full refund."
    if order["delivered"]:
        if order["ordered_days_ago"] <= 30:
            return f"Order {order_id.upper()} was delivered and is within the 30-day window, so it IS eligible for a full refund."
        return f"Order {order_id.upper()} was delivered over 30 days ago; NOT eligible for a standard refund. Offer store credit."
    return f"Order {order_id.upper()} has shipped but not yet delivered. The customer can return it within 30 days of arrival."


@tool
def search_faq(query: str) -> str:
    """Search the help center for shipping, returns, sizing, and payment info."""
    q = query.lower()
    for keywords, answer in _FAQ:
        if any(k in q for k in keywords):
            return answer
    return "No exact FAQ match. We ship in 2-4 business days locally, accept returns within 30 days, and sizing runs true to size."


@tool
def escalate_to_human(reason: str) -> str:
    """Escalate to a human agent and open a support ticket. Use when the customer is upset or the issue is complex."""
    ticket_id = f"TCK-{abs(hash(reason)) % 90000 + 10000}"
    return f"Escalated to a human agent. Ticket {ticket_id} created: '{reason}'. A specialist replies within 24 hours."


SUPPORT_TOOLS = [lookup_order, check_refund_eligibility, search_faq, escalate_to_human]
print(f"Defined {len(SUPPORT_TOOLS)} tools.")

## Step 3b - Build the LangGraph agent

We use LangGraph's prebuilt ReAct agent: the LLM reasons, calls tools, sees the results, and replies.

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

SYSTEM_PROMPT = (
    "You are Sunny, the customer-support assistant for Sunrise Outfitters, an online "
    "outdoor-apparel retailer in Singapore. Be warm and concise. Use the tools to look "
    "up real order details before answering; never invent statuses, tracking, or prices. "
    "Check eligibility before promising refunds. Escalate to a human if the customer is "
    "upset or you cannot resolve the issue. Keep replies under ~120 words."
)

def build_agent(model="gpt-4o-mini", temperature=0.0):
    llm = ChatOpenAI(model=model, temperature=temperature)
    return create_react_agent(llm, tools=SUPPORT_TOOLS, prompt=SYSTEM_PROMPT)

def run_agent(agent, user_message):
    result = agent.invoke({"messages": [HumanMessage(content=user_message)]})
    for m in reversed(result["messages"]):
        if isinstance(m, AIMessage) and m.content:
            return m.content
    return result["messages"][-1].content

agent = build_agent()
print("Agent ready.")

### Run the agent once (no tracing yet)

In [ ]:
print(run_agent(agent, "Where is my order A1001?"))

## Step 4 - Add Arize tracing

Now we register the Arize tracer and instrument LangChain. LangGraph runs on LangChain runnables, so this single instrumentor captures the **whole graph**: agent reasoning, every LLM call, and every tool call.

Get your **Space ID** and **API key** from [app.arize.com](https://app.arize.com) -> Settings -> Space API Keys.

In [ ]:
from arize.otel import register
from openinference.instrumentation.langchain import LangChainInstrumentor

os.environ["ARIZE_SPACE_ID"] = getpass("Arize Space ID: ")
os.environ["ARIZE_API_KEY"] = getpass("Arize API key: ")

tracer_provider = register(
    space_id=os.environ["ARIZE_SPACE_ID"],
    api_key=os.environ["ARIZE_API_KEY"],
    project_name="arize-singapore-workshop",
)
LangChainInstrumentor().instrument(tracer_provider=tracer_provider)
print("Tracing enabled -> project 'arize-singapore-workshop'.")

### Re-run the agent - now it's traced

Run a few queries, then open the `arize-singapore-workshop` project in Arize to see the traces (with tool calls and their inputs/outputs).

In [ ]:
for q in [
    "Can I get a refund on order A1002?",
    "How long does shipping take to Singapore?",
    "Order A1003 still hasn't shipped and I'm frustrated!",
]:
    print(f"Q: {q}")
    print(f"A: {run_agent(agent, q)}\n")

## Step 5 - Chat with the agent in a Gradio UI

This launches an interactive chat UI right inside the notebook (and prints a public share link). Every message is traced into Arize.

In [ ]:
import gradio as gr

def respond(message, history):
    try:
        return run_agent(agent, message)
    except Exception as e:
        return f"Sorry, something went wrong: {e}"

demo = gr.ChatInterface(
    fn=respond,
    title="Sunrise Outfitters Support",
    description="Ask about orders A1001, A1002, A1003, shipping, returns, or sizing.",
    examples=[
        "Where is my order A1001?",
        "Can I get a refund on order A1002?",
        "How long does shipping take to Singapore?",
        "Order A1003 hasn't arrived and I'm frustrated. Help!",
    ],
)
demo.launch(share=True, debug=False)

## Step 6 - Evaluate the agent

Two complementary approaches.

### 6a) Online LLM-as-a-judge (in the Arize UI)
In the Arize UI, on the `arize-singapore-workshop` project, add an **online evaluation** (LLM as a judge) such as *"Did the agent resolve the customer's request?"* or a hallucination check. It runs automatically over the traces you just generated. This is the fastest way to score live traffic - no code required.

### 6b) Offline experiment (with the `ax` CLI)
Below we build a small dataset, run the agent over it, grade each reply with an LLM judge, and create an **experiment** in Arize so you can compare versions over time.

In [ ]:
# A small evaluation set: question + expected behavior
eval_examples = [
    {"input": "Where is my order A1001?", "expected_behavior": "Looks up A1001; reports it shipped via DHL with tracking DHL-SG-77123."},
    {"input": "Can I get a refund on order A1002?", "expected_behavior": "Checks A1002 (delivered, 20 days ago) and confirms it IS eligible for a full refund."},
    {"input": "How long does shipping take within Singapore?", "expected_behavior": "Uses FAQ; answers 2-4 business days locally."},
    {"input": "Do your hiking boots run true to size?", "expected_behavior": "Uses FAQ; explains boots run about half a size large."},
    {"input": "Can you check order A9999 for me?", "expected_behavior": "Finds no order A9999 and asks the customer to verify the ID."},
]
len(eval_examples)

In [ ]:
# Run the agent and grade each reply with an LLM judge
import json
from openai import OpenAI

judge_client = OpenAI()
JUDGE_PROMPT = (
    "You are grading a customer-support agent's reply.\n\n"
    "Customer question:\n{question}\n\nExpected behavior:\n{expected}\n\n"
    "Agent reply:\n{output}\n\n"
    "Respond with ONLY JSON: "
    '{{"label": "correct" | "incorrect", "score": <0.0-1.0>, "explanation": "<one sentence>"}}'
)

def judge(question, expected, output):
    prompt = JUDGE_PROMPT.format(question=question, expected=expected, output=output)
    resp = judge_client.chat.completions.create(
        model="gpt-4o-mini", temperature=0.0,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
    )
    return json.loads(resp.choices[0].message.content)

results = []
for ex in eval_examples:
    out = run_agent(agent, ex["input"])
    ev = judge(ex["input"], ex["expected_behavior"], out)
    results.append({**ex, "output": out, "evaluation": ev})
    print(f"{ev['label']:>9} ({ev['score']:.2f})  {ex['input']}")

avg = sum(r["evaluation"]["score"] for r in results) / len(results)
print(f"\nAverage correctness: {avg:.2f}")

#### Push the results to Arize as a dataset + experiment (ax CLI)

The cells below use the `ax` CLI from inside Colab. This creates a versioned **dataset** and an **experiment** you can browse in the Arize UI. (You can also do everything in the repo locally - see `evals/README.md`.)

In [ ]:
%pip install -q arize-ax-cli
import os
# The ax CLI reads ARIZE_API_KEY from the environment (set in Step 4).
!ax --version

In [ ]:
# 1) Write the dataset file and create the dataset in Arize
import json
with open("dataset.json", "w") as f:
    json.dump([{"input": r["input"], "expected_behavior": r["expected_behavior"]} for r in results], f)

!ax datasets create --name support-eval-v1 --space-id "$ARIZE_SPACE_ID" --file dataset.json -o json

In [ ]:
# 2) Paste the DATASET_ID printed above, then export to get example IDs
DATASET_ID = "PASTE_DATASET_ID_HERE"
!ax datasets export {DATASET_ID} --stdout > exported_examples.json

# 3) Build a runs file linking each example_id to the agent output + eval
import json
examples = json.load(open("exported_examples.json"))
by_input = {r["input"]: r for r in results}
runs = []
for ex in examples:
    r = by_input.get(ex.get("input"))
    if not r:
        continue
    ev = r["evaluation"]
    runs.append({
        "example_id": ex["id"],
        "output": r["output"],
        "evaluations": {"correctness": {"label": ev["label"], "score": float(ev["score"]), "explanation": ev.get("explanation", "")}},
        "metadata": {"model": "gpt-4o-mini"},
    })
json.dump(runs, open("runs.json", "w"), indent=2)
print(f"Wrote {len(runs)} runs.")

In [ ]:
# 4) Create the experiment in Arize
!ax experiments create --name react-agent-baseline --dataset-id {DATASET_ID} --file runs.json -o json

## Recap

You built a LangGraph agent, traced it into Arize with one instrumentor, chatted with it via Gradio, and evaluated it both online (UI) and offline (ax CLI experiment).

Repo with the same code + a local Gradio app: **github.com/hakantekgul/arize-singapore-workshop**